# Day 14 — Causal Identification Feasibility

## Project

EconoCausal — Dynamic Pricing via Double Machine Learning

## Objective

Evaluate whether the available AmExpert 2019 Marketing Campaign datasets provide sufficient evidence to identify a treatment, define an outcome, and establish a defensible causal identification strategy.

## Tasks

* Review candidate treatment and outcome variables.
* Distinguish observed associations from verified treatment assignment.
* Identify potential confounders and variables that may occur after treatment.
* Examine the limitations of the available datasets.
* Document the assumptions and additional evidence required for causal estimation.

## Expected Outcome

A structured assessment of whether the available data supports a credible causal analysis using Double Machine Learning, together with the limitations and unresolved questions.


In [1]:
import pandas as pd
import numpy as np

# Load the relevant datasets
train = pd.read_csv("../data/raw/train.csv")
campaigns = pd.read_csv("../data/raw/campaign_data.csv")
customers = pd.read_csv("../data/raw/customer_demographics.csv")
transactions = pd.read_csv("../data/raw/customer_transaction_data.csv")

# Inspect dataset dimensions
print("===== DATASET SHAPES =====")
print("Training data:", train.shape)
print("Campaign data:", campaigns.shape)
print("Customer demographics:", customers.shape)
print("Transaction data:", transactions.shape)

# Inspect training columns
print("\n===== TRAINING DATA COLUMNS =====")
print(train.columns.tolist())

# Inspect campaign columns
print("\n===== CAMPAIGN DATA COLUMNS =====")
print(campaigns.columns.tolist())

# Inspect customer demographic columns
print("\n===== CUSTOMER DATA COLUMNS =====")
print(customers.columns.tolist())

# Inspect transaction columns
print("\n===== TRANSACTION DATA COLUMNS =====")
print(transactions.columns.tolist())

===== DATASET SHAPES =====
Training data: (78369, 5)
Campaign data: (28, 4)
Customer demographics: (760, 7)
Transaction data: (1324566, 7)

===== TRAINING DATA COLUMNS =====
['id', 'campaign_id', 'coupon_id', 'customer_id', 'redemption_status']

===== CAMPAIGN DATA COLUMNS =====
['campaign_id', 'campaign_type', 'start_date', 'end_date']

===== CUSTOMER DATA COLUMNS =====
['customer_id', 'age_range', 'marital_status', 'rented', 'family_size', 'no_of_children', 'income_bracket']

===== TRANSACTION DATA COLUMNS =====
['date', 'customer_id', 'item_id', 'quantity', 'selling_price', 'other_discount', 'coupon_discount']


In [2]:
# Day 14: Candidate Treatment Variable Investigation

print("===== TRAINING DATA SAMPLE =====")
display(train.head())

print("\n===== CAMPAIGN DATA SAMPLE =====")
display(campaigns.head())

print("\n===== TRANSACTION DATA SAMPLE =====")
display(transactions.head())

# Inspect unique values in selected training columns
print("\n===== TRAINING VARIABLE DISTRIBUTIONS =====")

for col in ["redemption_status", "campaign_id", "coupon_id"]:
    if col in train.columns:
        print(f"\n{col}:")
        print("Unique values:", train[col].nunique())
        display(train[col].value_counts().head(10))

# Inspect campaign characteristics
print("\n===== CAMPAIGN CHARACTERISTICS =====")

for col in campaigns.columns:
    print(f"\nColumn: {col}")
    print("Unique values:", campaigns[col].nunique())
    display(campaigns[col].value_counts().head(10))

# Inspect transaction columns
print("\n===== TRANSACTION COLUMN SUMMARY =====")
display(
    pd.DataFrame({
        "column": transactions.columns,
        "data_type": transactions.dtypes.astype(str).values,
        "unique_values": [
            transactions[col].nunique()
            for col in transactions.columns
        ]
    })
)

===== TRAINING DATA SAMPLE =====


,id,campaign_id,coupon_id,customer_id,redemption_status
0,1,13,27,1053,0
1,2,13,116,48,0
2,6,9,635,205,0
3,7,13,644,1050,0
4,9,8,1017,1489,0



===== CAMPAIGN DATA SAMPLE =====


,campaign_id,campaign_type,start_date,end_date
0,24,Y,21/10/13,20/12/13
1,25,Y,21/10/13,22/11/13
2,20,Y,07/09/13,16/11/13
3,23,Y,08/10/13,15/11/13
4,21,Y,16/09/13,18/10/13



===== TRANSACTION DATA SAMPLE =====


...

start_date
21/10/13    2
16/09/13    2
22/04/13    2
07/09/13    1
08/10/13    1
10/08/13    1
26/08/13    1
29/07/13    1
15/07/13    1
19/05/13    1
Name: count, dtype: int64


Column: end_date
Unique values: 26


end_date
18/10/13    2
18/01/13    2
20/12/13    1
22/11/13    1
16/11/13    1
15/11/13    1
04/10/13    1
27/09/13    1
30/08/13    1
16/08/13    1
Name: count, dtype: int64


===== TRANSACTION COLUMN SUMMARY =====


,column,data_type,unique_values
0,date,str,549
1,customer_id,int64,1582
2,item_id,int64,74063
3,quantity,int64,9252
4,selling_price,float64,4923
5,other_discount,float64,1418
6,coupon_discount,float64,232


In [3]:
# Day 14: Potential Confounder and Timing Investigation

print("===== CUSTOMER DEMOGRAPHIC VARIABLES =====")

customer_summary = pd.DataFrame({
    "variable": customers.columns,
    "data_type": customers.dtypes.astype(str).values,
    "missing_values": customers.isnull().sum().values,
    "unique_values": customers.nunique().values
})

display(customer_summary)


print("\n===== TRANSACTION DATE INFORMATION =====")

# Identify date-related columns
transaction_date_cols = [
    col for col in transactions.columns
    if "date" in col.lower() or "time" in col.lower()
]

print("Transaction date columns:", transaction_date_cols)

for col in transaction_date_cols:
    dates = pd.to_datetime(
        transactions[col],
        errors="coerce",
        dayfirst=True
    )

    print(f"\nColumn: {col}")
    print("Earliest date:", dates.min())
    print("Latest date:", dates.max())
    print("Missing/invalid dates:", dates.isna().sum())


print("\n===== CAMPAIGN DATE INFORMATION =====")

campaign_date_cols = [
    col for col in campaigns.columns
    if "date" in col.lower()
]

for col in campaign_date_cols:
    dates = pd.to_datetime(
        campaigns[col],
        errors="coerce",
        dayfirst=True
    )

    print(f"\nColumn: {col}")
    print("Earliest date:", dates.min())
    print("Latest date:", dates.max())
    print("Missing/invalid dates:", dates.isna().sum())

===== CUSTOMER DEMOGRAPHIC VARIABLES =====


,variable,data_type,missing_values,unique_values
0,customer_id,int64,0,760
1,age_range,str,0,6
2,marital_status,str,329,2
3,rented,int64,0,2
4,family_size,str,0,5
5,no_of_children,str,538,3
6,income_bracket,int64,0,12



===== TRANSACTION DATE INFORMATION =====
Transaction date columns: ['date']

Column: date
Earliest date: 2012-01-02 00:00:00
Latest date: 2013-12-06 00:00:00
Missing/invalid dates: 821695

===== CAMPAIGN DATE INFORMATION =====

Column: start_date
Earliest date: 2012-08-12 00:00:00
Latest date: 2013-10-21 00:00:00
Missing/invalid dates: 0

Column: end_date
Earliest date: 2012-09-21 00:00:00
Latest date: 2013-12-20 00:00:00
Missing/invalid dates: 0


C:\Users\KESHAV DEV GAUR\AppData\Local\Temp\ipykernel_29568\1222291427.py:46: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(
C:\Users\KESHAV DEV GAUR\AppData\Local\Temp\ipykernel_29568\1222291427.py:46: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(


In [4]:
# Day 14: Transaction Timing Investigation

print("===== CAMPAIGN DATE INFORMATION =====")

display(campaigns.head())

print("\nCampaign columns:")
print(campaigns.columns.tolist())

print("\n===== TRANSACTION DATE INFORMATION =====")

print("Transaction columns:")
print(transactions.columns.tolist())

# Identify potential date columns
transaction_date_cols = [
    col for col in transactions.columns
    if "date" in col.lower() or "time" in col.lower()
]

campaign_date_cols = [
    col for col in campaigns.columns
    if "date" in col.lower() or "time" in col.lower()
]

print("\nTransaction date columns:", transaction_date_cols)
print("Campaign date columns:", campaign_date_cols)

# Inspect sample transaction dates
for col in transaction_date_cols:
    print(f"\nSample values from {col}:")
    display(transactions[col].dropna().head(10))

# Inspect campaign start and end dates
for col in campaign_date_cols:
    print(f"\nSample values from {col}:")
    display(campaigns[col].dropna().head(10))

===== CAMPAIGN DATE INFORMATION =====


,campaign_id,campaign_type,start_date,end_date
0,24,Y,21/10/13,20/12/13
1,25,Y,21/10/13,22/11/13
2,20,Y,07/09/13,16/11/13
3,23,Y,08/10/13,15/11/13
4,21,Y,16/09/13,18/10/13



Campaign columns:
['campaign_id', 'campaign_type', 'start_date', 'end_date']

===== TRANSACTION DATE INFORMATION =====
Transaction columns:
['date', 'customer_id', 'item_id', 'quantity', 'selling_price', 'other_discount', 'coupon_discount']

Transaction date columns: ['date']
Campaign date columns: ['start_date', 'end_date']

Sample values from date:


0    2012-01-02
1    2012-01-02
2    2012-01-02
3    2012-01-02
4    2012-01-02
5    2012-01-02
6    2012-01-02
7    2012-01-02
8    2012-01-02
9    2012-01-02
Name: date, dtype: str


Sample values from start_date:


0    21/10/13
1    21/10/13
2    07/09/13
3    08/10/13
4    16/09/13
5    16/09/13
6    10/08/13
7    26/08/13
8    29/07/13
9    15/07/13
Name: start_date, dtype: str


Sample values from end_date:


0    20/12/13
1    22/11/13
2    16/11/13
3    15/11/13
4    18/10/13
5    18/10/13
6    04/10/13
7    27/09/13
8    30/08/13
9    16/08/13
Name: end_date, dtype: str

In [5]:
# Day 14: Causal Identification Feasibility Assessment

assessment = pd.DataFrame([
    {
        "assessment_area": "Treatment definition",
        "finding": "A clearly verified treatment variable has not yet been established",
        "status": "Unresolved"
    },
    {
        "assessment_area": "Outcome definition",
        "finding": "redemption_status is a candidate observed outcome",
        "status": "Candidate identified"
    },
    {
        "assessment_area": "Treatment assignment",
        "finding": "Available identifier and campaign relationships do not establish assignment",
        "status": "Insufficient evidence"
    },
    {
        "assessment_area": "Customer confounders",
        "finding": "Demographic variables are potential covariates; timing and relevance require validation",
        "status": "Requires investigation"
    },
    {
        "assessment_area": "Transaction features",
        "finding": "Transaction variables require validation of meaning and timing relative to treatment",
        "status": "Requires investigation"
    },
    {
        "assessment_area": "Temporal ordering",
        "finding": "Campaign and transaction dates must be interpreted and aligned correctly",
        "status": "Requires investigation"
    },
    {
        "assessment_area": "Causal identification",
        "finding": "A defensible assignment mechanism and identification assumptions have not been established",
        "status": "Not established"
    },
    {
        "assessment_area": "Double Machine Learning",
        "finding": "Causal effect estimation should wait until treatment and identification are justified",
        "status": "Not ready"
    }
])

display(assessment)

# Save the assessment
assessment.to_csv(
    "../reports/day14_causal_identification_assessment.csv",
    index=False
)

print("Day 14 assessment saved successfully.")

,assessment_area,finding,status
0,Treatment definition,A clearly verified treatment variable has not ...,Unresolved
1,Outcome definition,redemption_status is a candidate observed outcome,Candidate identified
2,Treatment assignment,Available identifier and campaign relationship...,Insufficient evidence
3,Customer confounders,Demographic variables are potential covariates...,Requires investigation
4,Transaction features,Transaction variables require validation of me...,Requires investigation
5,Temporal ordering,Campaign and transaction dates must be interpr...,Requires investigation
6,Causal identification,A defensible assignment mechanism and identifi...,Not established
7,Double Machine Learning,Causal effect estimation should wait until tre...,Not ready


Day 14 assessment saved successfully.
